# Kaggriculture Tutorial

Farm your way to market dominance! Players harvest produce and animal products to sell in a dynamic market.

## Game Mechanics 
 
- **Farmer**: takes one action per turn over a season with 30 days and 24 turns per day (720 total).
- **Crops and animals**: each have their own seed cost, time to first yield, and total payout. 
- **Daily care**: plants need watering every day or they turn to weeds, animals need feeding or they escape.
- **Market prices**: move with supply, selling a product pushes its price down, and crops vary in how hard they crash from a glut.
- **Town shops** unlock over the season and steadily buy products, lifting prices over time.
- **Farm hands** can be hired for the day, with increasing costs for each hire per day. 
- **Farm expansion**: start with one quadrant of land and can buy the other three for an escalating fee.
- **Shed**: holds harvested goods but caps at 100 items, anything past that is discarded at end of day.
 - **Win condition**: whoever has the most money in the bank at the end of the season wins.

In [840]:
from kaggle_environments import make

env = make("kaggriculture", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

Environment: kaggriculture v0.1.0
Players: [2]
Max steps: 720


## Understanding the Observation

Each turn your agent receives an observation with:

- **`player`** — your player id (`0` or `1`).
- **`day`** / **`hour`** — the current in-game day (0-indexed) and turn within that day (0-indexed; there are `turnsPerDay` turns per day).
- **`farms`** — a list of both players' public farm state, indexed by player id. Each farm has:
    - `money` — current bank balance
    - `tiles` — a `boardSize × boardSize` grid indexed `tiles[y][x]`; each cell is `None` (empty), `"LOCKED"` (unowned quadrant), or a dict describing a `PLANT`, `WEED`, `COOP`, or `PASTURE` (see below)
    - `farmer` — `[x, y]` position of your main farmer
    - `hands` — `[x, y]` positions of any hired hands active today
    - `unlocked_quadrants` — subset of `["NW", "NE", "SW", "SE"]`
    - `hires_today` — number of hires already made today (drives the next `HIRE` price)
- **`market`** (shared) — `inventory` and current `prices` per product (`WHEAT, CARROT, TOMATO, STRAWBERRY, MELON, EGG, MILK, WOOL, FERTILIZER`).
- **`town`** (shared) — `unlocked_shops`, the list of shops currently generating per-tick demand.
- **`private`** — your own hidden state (not visible to your opponent):
  - `shed` — counts of every product and animal (`GOOSE`, `COW`, `SHEEP`) stored in your shed
  - `seeds` — seed counts per crop
  - `inventories` — per-unit carried inventories; `[0]` is your main farmer, `[1..]` are today's hired hands in order

Tile dicts come in a few shapes:

- **Plant**: `{kind: "PLANT", crop, planted_day, watered_today, consecutive_unwatered, yield_units, max_lifespan_step, fertilized_until_day}` — `consecutive_unwatered >= 2` turns the tile to a weed at end of day.
- **Weed**: `{kind: "WEED"}` — must be `DIG`-ed before the tile is usable again.
- **Coop / Pasture** (empty): `{kind: "COOP" | "PASTURE"}`.
- **Coop / Pasture** (occupied): adds `animal, placed_day, yield_units, fed_today, consecutive_unfed, cared_today, fertilizer_available, pending_care_bonus`. `consecutive_unfed >= 2` means the animal escapes.

Your agent returns a dict of actions for the farmer, any farm hands, and the market: `{"farmer": 'PASS', "hands": [], "market": []}`


In [841]:
# Run a quick game to see what the observation looks like
env = make("kaggriculture", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
obs = env.steps[1][0].observation  # step 1 = first action step
items = obs.market.prices.keys()
print(f"Player: {obs.player}")
print(f"Player {obs.player}'s Unlocked Farm Areas: {obs.farms[obs.player].unlocked_quadrants}")
for i in items:
    print(f"{i} Price: {obs.market.prices[i]}")

Player: 0
Player 0's Unlocked Farm Areas: ['NW']
WHEAT Price: 26
CARROT Price: 35
TOMATO Price: 60
STRAWBERRY Price: 128
MELON Price: 256
EGG Price: 50
MILK Price: 169
WOOL Price: 206
FERTILIZER Price: 100


In [842]:
from kaggle_environments import make


In [843]:
# ============================================================
# CROP DATA
# Static information provided by the Kaggriculture competition.
#
# These values describe how each crop behaves.
# Current market prices are NOT stored here because they
# change during the game and are provided by obs["market"].
# ============================================================

CROP_DATA = {

    # Wheat is a one-time crop.
    # Without fertilizer, its maximum yield is 4.
    # With fertilizer, its maximum yield is 6.
    "WHEAT": {
        "yield_type": "ONE_TIME",
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "max_yield": 6,
        "unfertilized_max_yield": 4,
        "yield_per_tile_day": 0.80,
    },

    # Carrot is a one-time crop.
    "CARROT": {
        "yield_type": "ONE_TIME",
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "max_yield": 4,
        "unfertilized_max_yield": 3,
        "yield_per_tile_day": 0.75,
    },

    # Tomato is an ongoing crop with four scheduled yields.
    "TOMATO": {
        "yield_type": "ONGOING",
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "max_yield_day": 11,
        "max_yield": 4,
        "subsequent_yield_interval": 1,
        "yield_per_tile_day": 0.33,
    },

    # Strawberry is an ongoing crop with four scheduled yields.
    "STRAWBERRY": {
        "yield_type": "ONGOING",
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "max_yield_day": 16,
        "max_yield": 4,
        "subsequent_yield_interval": 2,
        "yield_per_tile_day": 0.24,
    },

    # Melon is a one-time crop with a high base market price.
    "MELON": {
        "yield_type": "ONE_TIME",
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "max_yield": 6,
        "unfertilized_max_yield": 6,
        "yield_per_tile_day": 0.55,
    },
}


# ============================================================
# ANIMAL DATA
# Static information provided by the Kaggriculture competition.
#
# Each animal produces a specific product over time.
# Animal production continues as long as the animal survives.
# ============================================================

ANIMAL_DATA = {

    # Geese live on a COOP and produce eggs every day
    # after the initial production delay.
    "GOOSE": {
        "product": "EGG",
        "purchase_cost": 300,
        "base_price": 50,
        "first_yield_day": 4,
        "yield_interval_days": 1,
        "max_held": 4,
        "structure": "COOP",
        "yield_per_tile_day": 1.00,
    },

    # Cows live on a PASTURE and produce milk every two days.
    "COW": {
        "product": "MILK",
        "purchase_cost": 400,
        "base_price": 160,
        "first_yield_day": 8,
        "yield_interval_days": 2,
        "max_held": 6,
        "structure": "PASTURE",
        "yield_per_tile_day": 0.50,
    },

    # Sheep live on a PASTURE and produce wool every three days.
    "SHEEP": {
        "product": "WOOL",
        "purchase_cost": 500,
        "base_price": 200,
        "first_yield_day": 6,
        "yield_interval_days": 3,
        "max_held": 6,
        "structure": "PASTURE",
        "yield_per_tile_day": 0.33,
    },
}


# ============================================================
# PRODUCT DATA
# Describes where each market product comes from.
#
# The current market price is NOT stored here.
# It must always be read from:
# obs["market"]["prices"][product]
# ============================================================

PRODUCT_DATA = {

    # Crop products
    "WHEAT": {
        "source_type": "CROP",
        "source": "WHEAT",
    },

    "CARROT": {
        "source_type": "CROP",
        "source": "CARROT",
    },

    "TOMATO": {
        "source_type": "CROP",
        "source": "TOMATO",
    },

    "STRAWBERRY": {
        "source_type": "CROP",
        "source": "STRAWBERRY",
    },

    "MELON": {
        "source_type": "CROP",
        "source": "MELON",
    },

    # Animal products
    "EGG": {
        "source_type": "ANIMAL",
        "source": "GOOSE",
    },

    "MILK": {
        "source_type": "ANIMAL",
        "source": "COW",
    },

    "WOOL": {
        "source_type": "ANIMAL",
        "source": "SHEEP",
    },

    # Fertilizer is produced by all surviving animals.
    "FERTILIZER": {
        "source_type": "ANIMAL_BYPRODUCT",
        "source": "ALL_ANIMALS",
    },
}

In [844]:
def is_crop_ready_to_harvest(obs,tile):
    """
    Determine whether a crop has reached its configured harvest point.

    The decision is based on:
    - the tile being a plant;
    - the crop having available yield;
    - the crop's age;
    - the crop's configured max_yield_day.
    """

    if tile is None:
        return False

    if tile.get("kind") != "PLANT":
        return False
    crop_name = tile["crop"]
    crop = CROP_DATA[crop_name]

    age = obs["day"] - tile["planted_day"]
    return (tile.get("yield_units", 0) > 0 
        and
        age > crop["max_yield_day"]
    )

In [845]:
# ============================================================
# CURRENT EXPERIMENT
# ============================================================
# Crop currently being used by the experimental agent.
# Changing this value will allow the same agent logic
# to be tested with different crops later.

CROP = "CARROT"

# Get the static information for the selected crop.
crop = CROP_DATA[CROP]

print(f"Selected crop: {CROP}")
print(f"Seed cost: {crop['seed_cost']}")
print(f"Base price: {crop['base_price']}")
print(f"First yield day: {crop['first_yield_day']}")
print(f"Max yield day: {crop['max_yield_day']}")
print(f"Max yield: {crop['max_yield']}")

Selected crop: CARROT
Seed cost: 20
Base price: 35
First yield day: 2
Max yield day: 3
Max yield: 4


In [846]:
# Test the harvest decision at different crop ages.

tile = {
    "kind": "PLANT",
    "crop": "WHEAT",
    "planted_day": 0,
    "yield_units": 4,
}

for day in range(0, 6):
    obs_test = {"day": day}

    print(
        f"Day {day} | "
        f"Age: {obs_test["day"] - tile["planted_day"]} | "
        f"Ready: {is_crop_ready_to_harvest(obs_test, tile)}"
    )

Day 0 | Age: 0 | Ready: False
Day 1 | Age: 1 | Ready: False
Day 2 | Age: 2 | Ready: False
Day 3 | Age: 3 | Ready: False
Day 4 | Age: 4 | Ready: False
Day 5 | Age: 5 | Ready: True


In [847]:
def get_farmer_tile(obs):
    player = obs["player"]
    farm = obs["farms"][player]

    x, y = farm["farmer"]

    return farm["tiles"][y][x]

In [848]:
# Returns one movement action that reduces the Manhattan distance
# between the farmer's current position and the target position.
# The function moves horizontally first, then vertically.
# Returns None when the farmer is already at the target.
def step_toward(fx, fy, tx, ty):
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None


In [849]:
fx, fy = 4, 4
tx, ty = 2, 4

while (fx, fy) != (tx, ty):
    move = step_toward(fx, fy, tx, ty)

    print(f"Position: {(fx, fy)} | Action: {move}")

    if move == "WEST":
        fx -= 1
    elif move == "EAST":
        fx += 1
    elif move == "NORTH":
        fy -= 1
    elif move == "SOUTH":
        fy += 1

print(f"Reached target: {(fx, fy)}")

Position: (4, 4) | Action: WEST
Position: (3, 4) | Action: WEST
Reached target: (2, 4)


In [850]:
# Finds the best available farm task based on the current observation.
# Tasks are prioritized as harvest, water, and plant.
# Within the same task priority, the nearest tile is selected
# using Manhattan distance.
# The target is recalculated from the current state each turn.
def find_target(obs, have_seed=False):
    player = obs["player"]
    farm = obs["farms"][player]

    fx, fy = farm["farmer"]
    candidates = []

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):

            if not isinstance(tile, dict):
                if tile is None and have_seed:
                    candidates.append((x, y, "plant"))
                continue

            if tile.get("kind") != "PLANT":
                continue

            # A crop needs to be watered today.
            if not tile["watered_today"]:
                candidates.append((x, y, "water"))

            # The crop is ready to harvest.
            if is_crop_ready_to_harvest(obs, tile):
                candidates.append((x, y, "harvest"))

    if not candidates:
        return None

    priority = {
        "harvest": 0,
        "water": 1,
        "plant": 2,
    }

    candidates.sort(
        key=lambda c: (
            priority[c[2]],
            abs(c[0] - fx) + abs(c[1] - fy),
        )
    )

    return candidates[0]

In [851]:
target = find_target(observations[-1], True)

print(target)

(4, 4, 'plant')


In [852]:



observations = []


def agent(obs):
    observations.append(obs)
    tile = get_farmer_tile(obs)
    target = find_target(obs)

    print(
        f"DEBUG TARGET | "
        f"Day: {obs['day']} | "
        f"Hour: {obs['hour']} | "
        f"Position: {obs['farms'][obs['player']]['farmer']} | "
        f"Target: {target}"
    )

    if target is not None:
        tx, ty, purpose = target

        fx, fy = obs["farms"][obs["player"]]["farmer"]

        move = step_toward(fx, fy, tx, ty)

        print(f"DEBUG MOVEMENT | Target: {target}")

        if move is not None:
            print(f"DEBUG MOVE | Action: {move}")

            return {
                "farmer": [move],
                "hands": [],
                "market": [],
            }

        
    if obs["day"] == 0 and obs["hour"] == 0:
        return {
            "farmer": ["PASS"],
            "hands": [],
            "market": [
                ["BUY_SEED", CROP , 1]
            ],
        }
    if obs["day"] == 0 and obs["hour"] == 1:
        return {
            "farmer": ["PLANT", CROP],
            "hands": [],
            "market": [],
        }
    if obs["day"] == 0 and obs["hour"] == 2:
        return {
            "farmer": ["WATER"],
            "hands": [],
            "market": [],
        }

        
    if is_crop_ready_to_harvest(obs, tile):
        return {
            "farmer": ["HARVEST"],
            "hands": [],
            "market": [],
    }
    # Drop harvested products into the shed when the main farmer
    # is carrying the selected crop and is standing on a tile
    # adjacent to the shed.
    #
    # The farmer's inventory is stored at private["inventories"][0].
    # The shed is not a tile, so its four adjacent positions are
    # (4,4), (4,5), (5,4), and (5,5) on the default 10x10 board.
    # This condition is state-based rather than tied to a specific
    # day or hour, making it reusable for different crops.
    if (
        obs['private']['inventories'][0].get(CROP, 0) > 0
        and {tuple(obs['farms'][obs['player']]['farmer']) 
        in {(4, 4), (4, 5), (5, 4), (5, 5)}}
        ):
        return {
            "farmer": ["DROP"],
            "hands": [],
            "market": [],
        }
    if obs["private"]["shed"].get(CROP, 0) > 0:
        return {
            "farmer": ["PASS"],
            "hands": [],
            "market": [
                ["SELL", CROP, obs["private"]["shed"].get(CROP, 0)]
            ],
        }
    if obs["hour"] == 0:
        return {
            "farmer": ["WATER"],
            "hands": [],
            "market": [],
        }

    return {
        "farmer": ["PASS"],
        "hands": [],
        "market": [],
    }




In [865]:
observations = []


def agent(obs):
    observations.append(obs)

    tile = get_farmer_tile(obs)
    target = find_target(obs, have_seed=False)

    print(
        f"DEBUG TARGET | "
        f"Day: {obs['day']} | "
        f"Hour: {obs['hour']} | "
        f"Position: {obs['farms'][obs['player']]['farmer']} | "
        f"Target: {target}"
    )

    # Move toward the selected target one tile at a time.
    if target is not None:
        tx, ty, purpose = target
        fx, fy = obs["farms"][obs["player"]]["farmer"]

        move = step_toward(fx, fy, tx, ty)

        print(f"DEBUG MOVEMENT | Target: {target}")

        if move is not None:
            print(f"DEBUG MOVE | Action: {move}")

            return {
                "farmer": [move],
                "hands": [],
                "market": [],
            }

    # Execute the task after reaching the target.
    if target is not None:
        tx, ty, purpose = target

        if purpose == "harvest":
            return {
                "farmer": ["HARVEST"],
                "hands": [],
                "market": [],
            }

        if purpose == "water":
            return {
                "farmer": ["WATER"],
                "hands": [],
                "market": [],
            }

    return {
        "farmer": ["PASS"],
        "hands": [],
        "market": [],
    }

In [892]:
def agent(obs):
    return {
        "farmer": ["EAST"],
        "hands": [],
        "market": [],
    }

In [893]:
env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 240,
    },
)

In [894]:

env.run([agent, "random"])

[[{'action': {'farmer': ['PASS'], 'hands': [], 'market': []},
   'reward': 0,
   'info': {},
   'observation': {'remainingOverageTime': 60,
    'step': 0,
    'player': 0,
    'farms': [{'money': 3000.0,
      'tiles': [[None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
 

In [897]:
print(agent)
print(agent.__name__)
print(agent.__code__.co_firstlineno)
print(env)
print(env.steps[0])
print(len(env.steps))

<function agent at 0x0000014358B28C20>
agent
1
[{'action': {'farmer': ['PASS'], 'hands': [], 'market': []}, 'reward': 0, 'info': {}, 'observation': {'remainingOverageTime': 60, 'step': 0, 'player': 0, 'farms': [{'money': 3000.0, 'tiles': [[None, None, None, None, None, 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], [None, None, None, None, None, 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], [None, None, None, None, None, 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], [None, None, None, None, None, 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], [None, None, None, None, None, 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], ['LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], ['LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], ['LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOCKED'], ['LOCKED', 'LOCKED', 'LOCKED', 'LOCKED', 'LOC

In [900]:
TEST_STEPS = 240

env_test = make(
    "kaggriculture",
    configuration={
        "episodeSteps": TEST_STEPS
    },
    debug=True
)

In [ ]:
def test_agent(obs):
    return {
        "farmer": ["EAST"],
        "hands": [],
        "market": [],
    }

{'farmer': ['EAST'], 'hands': [], 'market': []}


In [902]:
env_test.run([test_agent, "random"])

[[{'action': {'farmer': ['PASS'], 'hands': [], 'market': []},
   'reward': 0,
   'info': {},
   'observation': {'remainingOverageTime': 60,
    'step': 0,
    'player': 0,
    'farms': [{'money': 3000.0,
      'tiles': [[None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED'],
       [None,
        None,
        None,
        None,
        None,
        'LOCKED',
        'LOCKED',
        'LOCKED',
        'LOCKED',
 

In [903]:
print(len(env_test.steps))
print(env_test.steps[0][0]["action"])
print(env_test.steps[0][1]["action"])

240
{'farmer': ['PASS'], 'hands': [], 'market': []}
{'farmer': ['PASS'], 'hands': [], 'market': []}


In [895]:
obs = env.reset()
print(agent(obs))

{'farmer': ['EAST'], 'hands': [], 'market': []}


In [855]:
def summarize_observation(obs):
    player = obs["player"]
    farm = obs["farms"][player]
    private = obs["private"]
    market = obs["market"]
    town = obs["town"]

    return {
        "day": obs["day"],
        "hour": obs["hour"],
        "money": farm["money"],
        "farmer_position": farm["farmer"],
        "hands": len(farm["hands"]),
        "unlocked_quadrants": farm["unlocked_quadrants"],
        "seeds": private["seeds"],
        "shed": private["shed"],
        "market_prices": market["prices"],
        "unlocked_shops": town["unlocked_shops"],
    }

In [856]:
summary = summarize_observation(observations[25])
summary

{'day': 1,
 'hour': 1,
 'money': 2980.0,
 'farmer_position': [4, 4],
 'hands': 0,
 'unlocked_quadrants': ['NW'],
 'seeds': {'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWBERRY': 0, 'MELON': 0},
 'shed': {'WHEAT': 0,
  'CARROT': 0,
  'TOMATO': 0,
  'STRAWBERRY': 0,
  'MELON': 0,
  'EGG': 0,
  'MILK': 0,
  'WOOL': 0,
  'FERTILIZER': 0,
  'GOOSE': 0,
  'COW': 0,
  'SHEEP': 0},
 'market_prices': {'WHEAT': 26,
  'CARROT': 35,
  'TOMATO': 60,
  'STRAWBERRY': 132,
  'MELON': 260,
  'EGG': 50,
  'MILK': 172,
  'WOOL': 209,
  'FERTILIZER': 100},
 'unlocked_shops': []}

In [857]:
import pandas as pd
summaries = [
    summarize_observation(obs)
    for obs in observations
]

df = pd.DataFrame(summaries)

df

,day,hour,money,farmer_position,hands,unlocked_quadrants,seeds,shed,market_prices,unlocked_shops
0,0,0,3000.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 25, 'CARROT': 35, 'TOMATO': 60, 'STR...",[]
1,0,1,2980.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 1, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 26, 'CARROT': 35, 'TOMATO': 60, 'STR...",[]
2,0,2,2980.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 26, 'CARROT': 35, 'TOMATO': 60, 'STR...",[]
3,0,3,2980.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 26, 'CARROT': 35, 'TOMATO': 60, 'STR...",[]
4,0,4,2980.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 26, 'CARROT': 35, 'TOMATO': 60, 'STR...",[]
...,...,...,...,...,...,...,...,...,...,...
234,9,18,3088.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 32, 'CARROT': 39, 'TOMATO': 66, 'STR...","[FARMERS_MARKET, SMOOTHIE_SHOP, YARN_STORE]"
235,9,19,3088.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 32, 'CARROT': 39, 'TOMATO': 66, 'STR...","[FARMERS_MARKET, SMOOTHIE_SHOP, YARN_STORE]"
236,9,20,3088.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 32, 'CARROT': 39, 'TOMATO': 66, 'STR...","[FARMERS_MARKET, SMOOTHIE_SHOP, YARN_STORE]"
237,9,21,3088.0,"[4, 4]",0,[NW],"{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 0, 'CARROT': 0, 'TOMATO': 0, 'STRAWB...","{'WHEAT': 32, 'CARROT': 39, 'TOMATO': 66, 'STR...","[FARMERS_MARKET, SMOOTHIE_SHOP, YARN_STORE]"


In [858]:
def get_farmer_tile(obs):
    player = obs["player"]
    farm = obs["farms"][player]

    x, y = farm["farmer"]

    return farm["tiles"][y][x]

In [859]:
def print_daily_crop_history(observations):
    for obs in observations:
        if obs["hour"] != 0:
            continue

        player = obs["player"]
        farm = obs["farms"][player]
        tile = get_farmer_tile(obs)

        crop = tile.get("crop") if isinstance(tile, dict) else None

        print(
            f"Day {obs['day']} | "
            f"Money: {farm['money']:.0f} | "
            f"Crop: {crop} | "
            f"Age: {obs['day'] - tile['planted_day'] if crop else '-'} | "
            f"Yield: {tile['yield_units'] if crop else '-'} | "
            f"Price: {obs['market']['prices'].get(crop, '-') if crop else '-'}"
        )

In [860]:
print_daily_crop_history(observations)

Day 0 | Money: 3000 | Crop: None | Age: - | Yield: - | Price: -
Day 1 | Money: 2980 | Crop: CARROT | Age: 1 | Yield: 1 | Price: 35
Day 2 | Money: 2980 | Crop: CARROT | Age: 2 | Yield: 1 | Price: 35
Day 3 | Money: 2980 | Crop: CARROT | Age: 3 | Yield: 2 | Price: 35
Day 4 | Money: 2980 | Crop: CARROT | Age: 4 | Yield: 3 | Price: 36
Day 5 | Money: 3088 | Crop: None | Age: - | Yield: - | Price: -
Day 6 | Money: 3088 | Crop: None | Age: - | Yield: - | Price: -
Day 7 | Money: 3088 | Crop: None | Age: - | Yield: - | Price: -
Day 8 | Money: 3088 | Crop: None | Age: - | Yield: - | Price: -
Day 9 | Money: 3088 | Crop: None | Age: - | Yield: - | Price: -


In [861]:
for obs in observations:
    if obs["hour"] == 0:
        tile = get_farmer_tile(obs)

        print(
            f"Day {obs['day']} | "
            f"Money: {obs['farms'][obs['player']]['money']} | "
            f"Carrot price: {obs['market']['prices']['CARROT']} | "
            f"Tile: {tile}"
        )

Day 0 | Money: 3000.0 | Carrot price: 35 | Tile: None
Day 1 | Money: 2980.0 | Carrot price: 35 | Tile: {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 1, 'max_lifespan_step': 96, 'fertilized_until_day': -1}
Day 2 | Money: 2980.0 | Carrot price: 35 | Tile: {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 1, 'max_lifespan_step': 96, 'fertilized_until_day': -1}
Day 3 | Money: 2980.0 | Carrot price: 35 | Tile: {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 2, 'max_lifespan_step': 96, 'fertilized_until_day': -1}
Day 4 | Money: 2980.0 | Carrot price: 36 | Tile: {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 0, 'watered_today': False, 'consecutive_unwatered': 0, 'yield_units': 3, 'max_lifespan_step': 96, 'fertilized_until_day': -1}
Day 5 | Money: 3088.0 | Carrot price: 36 |

In [862]:
for step_index in range(len(env.steps)):
    for player_state in env.steps[step_index]:
        if player_state.observation["day"] == 5 and player_state.observation["hour"] in [0, 1, 2,3,4,5]:
            print(
                f"Step: {step_index} | "
                f"Day: {player_state.observation['day']} | "
                f"Hour: {player_state.observation['hour']} | "
                f"Action: {player_state.action}"
            )

Step: 120 | Day: 5 | Hour: 0 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 120 | Day: 5 | Hour: 0 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 121 | Day: 5 | Hour: 1 | Action: {'farmer': ['WATER'], 'hands': [], 'market': []}
Step: 121 | Day: 5 | Hour: 1 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 122 | Day: 5 | Hour: 2 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 122 | Day: 5 | Hour: 2 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 123 | Day: 5 | Hour: 3 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 123 | Day: 5 | Hour: 3 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 124 | Day: 5 | Hour: 4 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 124 | Day: 5 | Hour: 4 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 125 | Day: 5 | Hour: 5 | Action: {'farmer': ['PASS'], 'hands': [], 'market': []}
Step: 125 | Day: 5 | Hour: 5 | Action: {'f

In [863]:
for step_index in range(118, 126):
    for player_state in env.steps[step_index]:
        obs = player_state.observation

        if obs["player"] == 0:
            print(
                f"Step {step_index} | "
                f"Day {obs['day']} | "
                f"Hour {obs['hour']} | "
                f"Position {obs['farms'][0]['farmer']} | "
                f"Action {player_state.action} | "
                f"Inventory {obs['private']['inventories'][0]}"
            )

Step 118 | Day 4 | Hour 22 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 119 | Day 4 | Hour 23 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 120 | Day 5 | Hour 0 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 121 | Day 5 | Hour 1 | Position [4, 4] | Action {'farmer': ['WATER'], 'hands': [], 'market': []} | Inventory {}
Step 122 | Day 5 | Hour 2 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 123 | Day 5 | Hour 3 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 124 | Day 5 | Hour 4 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}
Step 125 | Day 5 | Hour 5 | Position [4, 4] | Action {'farmer': ['PASS'], 'hands': [], 'market': []} | Inventory {}


## What's wrong with this agent?

The melon agent has a few problems:
  - It never hires farm hands or buys more land, so it's stuck with one farmer working a single quadrant.     
  - It only grows melons, so when other crops are fetching a higher price it has nothing to sell.
  - It never fertilizes its plants, leaving a yield bonus on the table.                           
  - When it does sell, it dumps the entire inventory in one order, which can crash the melon price below the       
  threshold partway through the sale.                                                                         
                                      
Now it's your turn to make improvements!

## Making a submission

You can either submit a main.py, a tar.gz (or zip) with a main.py in it, or submit a notebook with a main.py or submission.tar.gz

There are three ways to subit.
1. using the [Submit Agent](https://www.kaggle.com/competitions/kaggriculture-gdm-internal) button on the homepage and uploading the file
2. using the Kaggle CLI (as described in agents.py in the competition dataset)
3. submitting a notebook with a submission.py or submission.tar.gz

In [864]:
%%writefile submission.py
from kaggle_environments.envs.kaggriculture.kaggriculture import CROPS

MELON_SEED_COST = CROPS["MELON"]["seed"]
MELON_MAX_YIELD_DAY = CROPS["MELON"]["max_yield_day"]
SELL_THRESHOLD = 200

def _step_toward(fx, fy, tx, ty):
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None

def _find_target_tile(farm, board_size, have_seed):
    fx, fy = farm["farmer"]
    candidates = []
    for y in range(board_size):
        for x in range(board_size):
            tile = farm["tiles"][y][x]
            if isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile["crop"] == "MELON":
                purpose = None
                # Harvest if ripe.
                age_ok = tile["yield_units"] > 0
                if age_ok and tile.get("planted_day") is not None:
                    purpose = "harvest"
                if not tile["watered_today"]:
                    purpose = "water" if purpose is None else purpose
                if purpose:
                    candidates.append((x, y, purpose))
            elif tile is None and have_seed:
                candidates.append((x, y, "plant"))

    if not candidates:
        return None

    priority = {"harvest": 0, "water": 1, "plant": 2}
    candidates.sort(key=lambda c: (priority[c[2]], abs(c[0] - fx) + abs(c[1] - fy)))
    return candidates[0]

def melon_maxxer(obs):
    farms = obs.get("farms", [])
    player = obs.get("player", 0)
    private = obs.get("private", {}) or {}
    if not farms or player >= len(farms):
        return {"farmer": ["PASS"], "hands": [], "market": []}

    farm = farms[player]
    board_size = len(farm["tiles"])
    fx, fy = farm["farmer"]
    tile = farm["tiles"][fy][fx]
    day = obs.get("day", 0)

    seeds = private.get("seeds", {})
    shed = private.get("shed", {})
    market_prices = (obs.get("market", {}) or {}).get("prices", {})
    melon_price = market_prices.get("MELON", 0)

    market = []

    # Sell melons only when the market is paying enough.
    melons_in_shed = shed.get("MELON", 0)
    if melons_in_shed > 0 and melon_price >= SELL_THRESHOLD:
        market.append(["SELL", "MELON", melons_in_shed])

    # Top up seed inventory so the next empty tile can be planted.
    if seeds.get("MELON", 0) == 0 and farm["money"] >= MELON_SEED_COST:
        market.append(["BUY_SEED", "MELON", 1])

    # Decide farmer action.
    farmer = ["PASS"]

    if isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile["crop"] == "MELON":
        age = day - tile["planted_day"]
        if age >= MELON_MAX_YIELD_DAY and tile["yield_units"] > 0:
            farmer = ["HARVEST"]
        elif not tile["watered_today"]:
            farmer = ["WATER"]
        else:
            target = _find_target_tile(farm, board_size, seeds.get("MELON", 0) > 0)
            if target:
                step = _step_toward(fx, fy, target[0], target[1])
                if step:
                    farmer = [step]
    elif tile is None and seeds.get("MELON", 0) > 0:
        farmer = ["PLANT", "MELON"]
    else:
        target = _find_target_tile(farm, board_size, seeds.get("MELON", 0) > 0)
        if target:
            step = _step_toward(fx, fy, target[0], target[1])
            if step:
                farmer = [step]

    return {"farmer": farmer, "hands": [], "market": market}

Overwriting submission.py


## Submit to competition

Now that we have a main.py, all you need to do is click "Submit to competition" on the right and watch your entry show up on the competition leaderboard! Best of luck!